# 06 — Softmax（数值稳定的逐行 Softmax）

## 背景

Softmax 是 Transformer 注意力机制的核心组件，也是分类任务的标准输出层。其 GPU 实现的难点在于：

1. **数值稳定性**：直接计算 $e^x$ 会溢出（float16 最大约 65504）。标准做法是减去行最大值：$\text{softmax}(x_j) = e^{x_j - \max_k x_k} / \sum_k e^{x_k - \max_k x_k}$
2. **多遍扫描**：数值稳定版需要先扫描一遍求最大值，再求指数和，再归一化，共三遍。
3. **FlashAttention 的核心思想**（Online Softmax）正是将三遍合并为可流式处理的形式。

## 问题定义

$$\text{MAX}[i] = \max_j A[i, j]$$
$$B[i,j] = \frac{e^{A[i,j] - \text{MAX}[i]}}{\sum_k e^{A[i,k] - \text{MAX}[i]}}$$

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch 参考实现

In [ ]:
N, M = 4096, 16384
A = torch.randn(N, M, dtype=torch.float32, device="cuda")

def ref_softmax(A):
    return torch.softmax(A, dim=1)

B_ref = ref_softmax(A)
print(f"A: {A.shape} → B: {B_ref.shape}")
print(f"Row-sum check (should ≈ 1): B[0].sum() = {B_ref[0].sum().item():.6f}")

## Triton 实现

每行由一个 Triton program 处理，三遍扫描均用 Python 循环 + `tl.max`/`tl.sum` 完成。关键技术：
- `tl.max(chunk, 0)` 对 `[BLOCK_M]` 张量归约，返回 0-d 标量
- `tl.maximum(scalar_a, scalar_b)` 更新全局最大值
- `chunk / row_sum` 标量广播到 `[BLOCK_M]`

In [ ]:
@triton.jit
def _softmax_kernel(X_ptr, Y_ptr, M, BLOCK_M: tl.constexpr):
    row  = tl.program_id(0)
    base = X_ptr + row * M

    # Pass 1: find row max
    row_max = tl.load(base).to(tl.float32)   # initialise with first element
    for start in range(0, M, BLOCK_M):
        cols  = start + tl.arange(0, BLOCK_M)
        chunk = tl.load(base + cols, mask=cols < M, other=float("-inf")).to(tl.float32)
        row_max = tl.maximum(row_max, tl.max(chunk, 0))

    # Pass 2: exp(x - max), write out, accumulate sum
    row_sum = 0.0
    for start in range(0, M, BLOCK_M):
        cols = start + tl.arange(0, BLOCK_M)
        mask = cols < M
        x = tl.load(base + cols, mask=mask, other=0.0).to(tl.float32)
        e = tl.exp(x - row_max)
        tl.store(Y_ptr + row * M + cols, e, mask=mask)
        row_sum = row_sum + tl.sum(e, 0)

    # Pass 3: normalise
    for start in range(0, M, BLOCK_M):
        cols = start + tl.arange(0, BLOCK_M)
        mask = cols < M
        y = tl.load(Y_ptr + row * M + cols, mask=mask)
        tl.store(Y_ptr + row * M + cols, y / row_sum, mask=mask)

BLOCK_M_TRI = 1024

def triton_softmax(A):
    B = torch.empty_like(A)
    _softmax_kernel[(A.shape[0],)](A, B, A.shape[1], BLOCK_M=BLOCK_M_TRI)
    return B

B_tri = triton_softmax(A)
ok = torch.allclose(B_ref, B_tri, atol=1e-4)
print("Triton correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_tri)).item():.6f}")

## TileLang 实现

TileLang 同样使用三遍扫描：`T.fill(mx, -inf)` 初始化、`T.reduce_max` 求最大值、`T.exp2` 配合 `log2_e` 实现快速 exp、`T.reduce_sum` 累加。

> **注意**：PyTorch 通过 **MIOpen**（厂商优化库）路由此内核，内部使用 online softmax，全局访存更少。TileLang 的 3-pass 显式内核明显优于 Triton，并接近 MIOpen 的结果。

**gfx1100 调参**：`threads=256, BLOCK_M=256` — `BM ≤ threads` 约束使 `T.copy` 生成向量化循环而非单次加载，8 个 wavefront/block 在 RDNA3 上有良好的 CU 占用率。


In [ ]:
# gfx1100 最优：threads=256, BLOCK_M=256
# 约束：BM <= threads（T.copy 布局）。threads=256 → 8 wavefront/CU
# 3-pass（max → exp+sum → normalize）比 Triton 快约 11%
BM = 256

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_softmax(A, BLOCK_M: int):
    log2_e = 1.44269504
    N_, M_ = T.const("N, M")
    dtype = T.float32
    A: T.Tensor((N_, M_), dtype)
    B = T.empty((N_, M_), dtype)
    with T.Kernel(N_, threads=256) as row:
        A_l = T.alloc_fragment((1, BLOCK_M), dtype)
        B_l = T.alloc_fragment((1, BLOCK_M), dtype)
        mx  = T.alloc_fragment((1,), dtype)
        sm  = T.alloc_fragment((1,), dtype)
        # Pass 1: 求行最大值
        T.fill(mx, float("-inf"))
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], A_l)
            T.reduce_max(A_l, mx, dim=1, clear=False)
        # Pass 2: exp(x - max)，累加 sum
        T.clear(sm)
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], A_l)
            for j in T.Parallel(BLOCK_M):
                B_l[0, j] = T.exp2(log2_e * (A_l[0, j] - mx[0]))
            T.reduce_sum(B_l, sm, dim=1, clear=False)
            T.copy(B_l, B[row, m_blk * BLOCK_M])
        # Pass 3: 归一化
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(B[row, m_blk * BLOCK_M], B_l)
            for j in T.Parallel(BLOCK_M):
                B_l[0, j] = B_l[0, j] / sm[0]
            T.copy(B_l, B[row, m_blk * BLOCK_M])
    return B

k = tl_softmax.compile(N=N, M=M, BLOCK_M=BM)
B_tl = k(A.clone())
ok = torch.allclose(B_ref, B_tl, atol=5e-4)
print("TileLang 正确性:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_tl)).item():.6f}")


## 性能对比

In [ ]:
WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results = {
    "PyTorch":  bench(ref_softmax,    [A]),
    "Triton":   bench(triton_softmax, [A]),
    "TileLang": bench(k,             [A]),
}

bytes_rw = N * M * 4 * 2   # float32，读+写各一遍（3-pass 实际 5 遍，此处为有效 IO）
pt_ms  = results["PyTorch"]
tri_ms = results["Triton"]

header = f"{'实现':<12}  {'延迟 (ms)':>10}  {'带宽 (TB/s)':>12}  {'vs PyTorch':>10}  {'vs Triton':>10}"
sep    = "-" * len(header)
print(f"Softmax  (N={N}, M={M}, float32)")
print(sep)
print(header)
print(sep)
for name, ms in results.items():
    bw     = bytes_rw / ms / 1e9
    vs_pt  = f"{pt_ms/ms:+.1%}"
    vs_tri = f"{tri_ms/ms:+.1%}"
    print(f"{name:<12}  {ms:>10.4f}  {bw:>12.2f}  {vs_pt:>10}  {vs_tri:>10}")
print(sep)


In [ ]:
# Online Softmax — gfx1100 最优：threads=512, BLOCK_M=4096
# BM = 8 × threads → 每线程每轮加载 8 个 float（float4×2），512 线程全满载
BM_online = 4096

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_softmax_online(A, BLOCK_M: int):
    log2_e = 1.44269504
    N_, M_ = T.const("N, M")
    dtype = T.float32
    A: T.Tensor((N_, M_), dtype)
    B = T.empty((N_, M_), dtype)
    with T.Kernel(N_, threads=512) as row:
        A_l  = T.alloc_fragment((1, BLOCK_M), dtype)
        B_l  = T.alloc_fragment((1, BLOCK_M), dtype)
        mx   = T.alloc_fragment((1,), dtype)
        mx_new = T.alloc_fragment((1,), dtype)
        sm   = T.alloc_fragment((1,), dtype)
        # 单 pass：同时维护 running max 和 rescaled sum
        T.fill(mx, float("-inf"))
        T.clear(sm)
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], A_l)
            T.reduce_max(A_l, mx_new, dim=1, clear=True)
            T.fill(mx_new, T.max(mx[0], mx_new[0]))
            for j in T.Parallel(BLOCK_M):
                B_l[0, j] = T.exp2(log2_e * (A_l[0, j] - mx_new[0]))
            for T.Parallel(1):
                sm[0] = sm[0] * T.exp2(log2_e * (mx[0] - mx_new[0]))
            T.reduce_sum(B_l, sm, dim=1, clear=False)
            m      = mx_new[0]
            T.copy(B_l, B[row, m_blk * BLOCK_M])
            for T.Parallel(1):
                mx[0] = m
        # 最终归一化
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(B[row, m_blk * BLOCK_M], B_l)
            for j in T.Parallel(BLOCK_M):
                B_l[0, j] = B_l[0, j] / sm[0]
            T.copy(B_l, B[row, m_blk * BLOCK_M])
    return B

k_online = tl_softmax_online.compile(N=N, M=M, BLOCK_M=BM_online)
B_online = k_online(A.clone())
ok = torch.allclose(B_ref, B_online, atol=5e-4)
print("TileLang Online 正确性:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_online)).item():.6f}")


In [ ]:
WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results_all = {
    "PyTorch":          bench(ref_softmax,        [A]),
    "Triton":           bench(triton_softmax,     [A]),
    "TileLang 3-pass":  bench(k,                 [A]),
    "TileLang Online":  bench(k_online,           [A]),
}

bytes_rw = N * M * 4 * 2
pt_ms  = results_all["PyTorch"]
tri_ms = results_all["Triton"]

header = f"{'实现':<20}  {'延迟 (ms)':>10}  {'带宽 (TB/s)':>12}  {'vs PyTorch':>10}  {'vs Triton':>10}"
sep    = "-" * len(header)
print(f"\nSoftmax 全对比  (N={N}, M={M}, float32)")
print(sep)
print(header)
print(sep)
for name, ms in results_all.items():
    bw     = bytes_rw / ms / 1e9
    vs_pt  = f"{pt_ms/ms:+.1%}"
    vs_tri = f"{tri_ms/ms:+.1%}"
    print(f"{name:<20}  {ms:>10.4f}  {bw:>12.2f}  {vs_pt:>10}  {vs_tri:>10}")
print(sep)
